# Read persist events directly from Kafka

This uses a separate consumer group, so it does not interfere with the application consumers. It reads `user-events` from the beginning and prints persist events that contain `city`.

In [ ]:
import json
import os
from pathlib import Path

from confluent_kafka import Consumer
from dotenv import load_dotenv

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(project_root / ".env")

consumer = Consumer(
    {
        "bootstrap.servers": os.getenv("KAFKA_BOOTSTRAP_SERVERS", "127.0.0.1:9092"),
        "group.id": "notebook-exploration",
        "auto.offset.reset": "earliest",
        "enable.auto.commit": False,
    }
)
consumer.subscribe([os.getenv("KAFKA_TOPIC", "user-events")])

In [ ]:
city_events = []

try:
    while len(city_events) < 10:
        message = consumer.poll(5.0)

        if message is None:
            print("No more messages received within 5 seconds.")
            break
        if message.error():
            raise RuntimeError(message.error())

        payload = json.loads(message.value().decode())
        if payload.get("type") == "persist" and "city" in payload:
            event = {
                "key": message.key().decode(),
                "partition": message.partition(),
                "offset": message.offset(),
                "city": payload["city"],
                "payload": payload,
            }
            city_events.append(event)
            print(event)
finally:
    consumer.close()

city_events